# Photo to 3D model + measurements

Runs the whole pipeline on Colab's free GPU and hands you a public URL for the web UI.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Without a GPU only the
`silhouette` placeholder backend will work.

Run the cells in order. Steps 6 and 7 are optional extras you can skip on the first pass.

| Step | What it does | Roughly |
|---|---|---|
| 1 | Check the GPU you were given | seconds |
| 2 | Cache model weights on Google Drive | seconds |
| 3 | Get the code | seconds |
| 4 | Install dependencies | 3-5 min |
| 5 | Install TripoSR | 3-6 min |
| 6 | *Optional:* install Hunyuan3D 2.1 (better quality) | 10-15 min |
| 7 | *Optional:* install UniDepth (measure without a marker) | 3-5 min |
| 8 | Print a marker | seconds |
| 9 | Launch the app | ~1 min to load weights |


## 1. Which GPU did we get?

Colab's free tier hands out a **T4 (about 15 GB, Turing)**. Turing has no bf16 and no
flash-attention, which is why this project uses TripoSR and Hunyuan3D 2.1 rather than
TRELLIS-2 - TRELLIS-2 loads in bf16 by default and needs manual patching to run here.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

import torch

print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"{props.name}: {props.total_memory / 2**30:.1f} GiB, compute capability {props.major}.{props.minor}")
    print("bf16 supported:", torch.cuda.is_bf16_supported())
    if props.total_memory / 2**30 < 20:
        print("\nUnder 20 GiB: keep hunyuan_texture off (the texture pass alone wants ~21 GB).")
else:
    print("\nNo GPU. Set Runtime > Change runtime type > T4 GPU, then rerun this cell.")

## 2. Cache model weights on Drive

Colab wipes its disk when the session ends, and these checkpoints are several GB. Pointing
the Hugging Face cache at Drive means you download them once instead of once per session.

This has to run **before** anything imports `transformers` or `huggingface_hub`, so keep it
above the install cells. Skip it if you would rather not mount Drive - everything still
works, it just re-downloads each session.

In [ ]:
USE_DRIVE_CACHE = True  # set False to skip mounting Drive

import os

if USE_DRIVE_CACHE:
    from google.colab import drive

    drive.mount("/content/drive")
    cache = "/content/drive/MyDrive/photo3d_cache"
    os.makedirs(cache, exist_ok=True)
    os.environ["HF_HOME"] = cache
    os.environ["TORCH_HOME"] = f"{cache}/torch"
    os.environ["U2NET_HOME"] = f"{cache}/rembg"  # where rembg keeps its ONNX model
    print("Weights will be cached in", cache)
else:
    print("Not using Drive: weights download again every session.")

## 3. Get the code

Two ways, pick one by setting `CODE_SOURCE`:

* `"github"` - clone the repo. Best for iterating: edit locally, push, re-run this cell.
* `"drive"` - copy the project folder from Drive. Use this if the code is not on GitHub;
  upload the whole `3d_model_generation` folder to your Drive first.

In [ ]:
CODE_SOURCE = "github"  # "github" or "drive"
GITHUB_REPO = "https://github.com/Dnyaneshwarigund12/3d_model_generation.git"
GITHUB_BRANCH = "main"
DRIVE_PROJECT = "/content/drive/MyDrive/3d_model_generation"
PROJECT_DIR = "/content/3d_model_generation"

import os
import shutil
import subprocess

if CODE_SOURCE == "github":
    if os.path.isdir(PROJECT_DIR):
        subprocess.run(["git", "-C", PROJECT_DIR, "pull", "--ff-only"], check=False)
    else:
        subprocess.run(
            ["git", "clone", "--branch", GITHUB_BRANCH, GITHUB_REPO, PROJECT_DIR],
            check=True,
        )
elif CODE_SOURCE == "drive":
    if not os.path.isdir(DRIVE_PROJECT):
        raise SystemExit(f"{DRIVE_PROJECT} not found. Upload the project folder to Drive.")
    if os.path.isdir(PROJECT_DIR):
        shutil.rmtree(PROJECT_DIR)
    shutil.copytree(DRIVE_PROJECT, PROJECT_DIR)
else:
    raise SystemExit("CODE_SOURCE must be 'github' or 'drive'.")

os.chdir(PROJECT_DIR)
print("Working in", os.getcwd())
print(sorted(os.listdir()))

## 4. Install dependencies

Note what is *not* here: torch. Colab ships a working CUDA build, and reinstalling it is the
most reliable way to break a session.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -r requirements-colab.txt

import cv2, numpy, trimesh, gradio

print("opencv", cv2.__version__, "| numpy", numpy.__version__)
print("trimesh", trimesh.__version__, "| gradio", gradio.__version__)
print("aruco available:", hasattr(cv2, "aruco"))

## 4b. Check the pipeline before touching any GPU model

The test suite covers the scale and measurement maths on CPU with synthetic images. If this
passes, anything that goes wrong later is the generation backend, not the pipeline.

In [ ]:
!python -m pytest tests -q

## 5. Install TripoSR (start here)

Fast, about 4 GB of VRAM, MIT licensed. Quality is the lowest of the candidates, so treat it
as proof that the flow works end to end, then move to step 6.

`torchmcubes` is a CUDA extension TripoSR uses for marching cubes, so it compiles on install
and takes a few minutes.

In [ ]:
import os
import subprocess

os.makedirs("third_party", exist_ok=True)
if not os.path.isdir("third_party/TripoSR/tsr"):
    subprocess.run(
        ["git", "clone", "-q", "https://github.com/VAST-AI-Research/TripoSR", "third_party/TripoSR"],
        check=True,
    )

!pip install -q omegaconf einops jaxtyping rembg onnxruntime
!pip install -q git+https://github.com/tatsy/torchmcubes.git

try:
    from torchmcubes import marching_cubes  # noqa: F401

    print("torchmcubes OK")
except Exception as exc:
    print("torchmcubes failed to build:", exc)
    print("TripoSR cannot extract a mesh without it. Use the hunyuan3d backend instead.")

## 6. Optional: Hunyuan3D 2.1, the quality backend

Slower (35-50 s per image in low-VRAM mode) and much better. Two custom extensions have to
compile, so expect 10-15 minutes and read any error output rather than assuming it worked.

Shape generation needs about 10 GB and fits a T4 comfortably. The texture pass wants ~21 GB
on its own, so leave `P3D_HUNYUAN_TEXTURE` off unless you are on a bigger card.

In [ ]:
import os
import subprocess

REPO = "third_party/Hunyuan3D-2.1"
if not os.path.isdir(f"{REPO}/hy3dshape"):
    subprocess.run(
        ["git", "clone", "-q", "https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1", REPO],
        check=True,
    )

# Installed one by one on purpose: the repo's own requirements.txt pins mirrors and
# builds that fail on Colab.
!pip install -q ninja pybind11 transformers diffusers accelerate safetensors omegaconf einops opencv-python scikit-image trimesh pygltflib xatlas pytorch-lightning torchmetrics timm onnxruntime torchdiffeq

!cd {REPO}/hy3dpaint/custom_rasterizer && pip install -e . --no-build-isolation -q
!cd {REPO}/hy3dpaint/DifferentiableRenderer && bash compile_mesh_painter.sh

import sys

sys.path[:0] = [f"{REPO}/hy3dshape", f"{REPO}/hy3dpaint", REPO]
try:
    from hy3dshape.pipelines import Hunyuan3DDiTFlowMatchingPipeline  # noqa: F401

    print("Hunyuan3D shape pipeline imports cleanly")
except Exception as exc:
    print("Import failed:", exc)
    print("Stay on the triposr backend and check the compile output above.")

## 7. Optional: UniDepth, for photos with no reference object

Only needed for the `estimate` scale source. Its output is a learned guess with a wide error
bar (20%+, worse on unusual objects), and it is labelled as an estimate everywhere it
appears. A printed marker costs one sheet of paper and is several times more accurate.

In [ ]:
!pip install -q timm
!pip install -q git+https://github.com/lpiccinelli-eth/UniDepth.git

try:
    from unidepth.models import UniDepthV2  # noqa: F401

    print("UniDepth OK")
except Exception as exc:
    print("UniDepth unavailable:", exc)
    print("The 'estimate' scale source will fail; the marker and card sources still work.")

## 8. Print a marker

This is what makes the measurements real rather than a guess. Download the PNG, print it at
**100% scale** (turn off "fit to page"), measure the printed black square with a ruler, and
use the measured value in the app.

Lay it flat next to the object, roughly in the same plane, with all four corners visible.

In [ ]:
!python tools/make_marker.py --mm 50 --pdf --out assets/markers/marker_50mm.png

from IPython.display import Image, display

display(Image("assets/markers/marker_50mm.png", width=320))

from google.colab import files

files.download("assets/markers/marker_50mm.pdf")

## 9. Launch the app

`share=True` gives a public `gradio.live` URL, valid for 72 hours and only while this cell
keeps running. Open it on your phone to upload photos directly.

Weights load on the first request, so the first model takes noticeably longer than the rest.

In [ ]:
GENERATOR = "triposr"  # "triposr", "hunyuan3d", or "silhouette" to test without a GPU

from app.config import Settings
from app.ui import build_ui

settings = Settings.from_env()
settings.generator = GENERATOR
settings.low_vram = True
settings.hunyuan_texture = False  # ~21 GB on its own; too much for a T4

build_ui(settings).queue().launch(share=True, show_error=True)

## 10. Or run it headless on one image

Useful for debugging: it prints every stage's timing and the full result, and writes
everything to `outputs/<run_id>/`.

In [ ]:
import json

from google.colab import files

from app.config import Settings
from app.pipeline import run

uploaded = files.upload()
photo = next(iter(uploaded))

settings = Settings.from_env()
settings.generator = "triposr"

result = run(photo, settings=settings, scale_source="marker", marker_mm=50.0)

print(result.summary)
print("timings:", result.timings_s)
for warning in result.warnings:
    print("warning:", warning)
print(json.dumps(result.measurements, indent=2))
print("outputs in", result.run_dir)

## 11. Are the numbers actually right?

The error percentages the app reports start as published estimates, not measurements of this
pipeline. To replace them with real ones: tape-measure 10-15 objects, photograph each with
the marker, fill in a CSV like `tools/validation_manifest.example.csv`, and run the cell
below. It prints measured error per tier.

In [ ]:
!python tools/validate.py --manifest tools/my_objects.csv --generator triposr